# 03 - Analise Detalhada de Inflacao (IPCA)

## Objetivo
Analisar profundamente a inflacao medida pelo IPCA ao longo do periodo.

## Fluxo
1. Carregar dados
2. Metricas de inflacao
3. Periodos de maior/menor inflacao
4. Tendencias e mudancas estruturais
5. Comparacoes temporais

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

## 1. Carregar dados

In [ ]:
caminho_dados = Path('../dados/brutos/indicadores_consolidados.csv')
df = pd.read_csv(caminho_dados)
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)

print(f'Dados carregados: {len(df)} observacoes')

## 2. Metricas principais de inflacao

In [ ]:
print('METRICAS DE INFLACAO (IPCA):')
print('=' * 60)

# IPCA mensal
ipca_mensal = df['ipca_mensal']
print(f'IPCA Mensal:')
print(f'  Media: {ipca_mensal.mean():.3f}%')
print(f'  Mediana: {ipca_mensal.median():.3f}%')
print(f'  Desvio Padrao: {ipca_mensal.std():.3f}%')
print(f'  Minimo: {ipca_mensal.min():.3f}% (data: {df.loc[ipca_mensal.idxmin(), "data"].strftime("%b/%Y")})')
print(f'  Maximo: {ipca_mensal.max():.3f}% (data: {df.loc[ipca_mensal.idxmax(), "data"].strftime("%b/%Y")})')
print()

# IPCA 12 meses
ipca_12m = df['ipca_acumulado_12_meses']
print(f'IPCA 12 Meses (Acumulado):')
print(f'  Media: {ipca_12m.mean():.2f}%')
print(f'  Mediana: {ipca_12m.median():.2f}%')
print(f'  Desvio Padrao: {ipca_12m.std():.2f}%')
print(f'  Minimo: {ipca_12m.min():.2f}% (data: {df.loc[ipca_12m.idxmin(), "data"].strftime("%b/%Y")})')
print(f'  Maximo: {ipca_12m.max():.2f}% (data: {df.loc[ipca_12m.idxmax(), "data"].strftime("%b/%Y")})')
print()

# Ultimo valor
print(f'Ultimos valores:')
print(f'  IPCA Mensal: {ipca_mensal.iloc[-1]:.3f}%')
print(f'  IPCA 12 Meses: {ipca_12m.iloc[-1]:.2f}%')

## 3. Evolucao temporal

In [ ]:
# Criar figura com 2 series
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# IPCA Mensal
ax1.plot(df['data'], df['ipca_mensal'], linewidth=2, color='#1f77b4', marker='o', markersize=4)
ax1.fill_between(df['data'], df['ipca_mensal'], alpha=0.3, color='#1f77b4')
ax1.axhline(y=df['ipca_mensal'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df["ipca_mensal"].mean():.3f}%')
ax1.set_title('IPCA Mensal ao Longo do Tempo', fontsize=14, fontweight='bold')
ax1.set_ylabel('IPCA (%)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# IPCA 12 meses
ax2.plot(df['data'], df['ipca_acumulado_12_meses'], linewidth=2, color='#ff7f0e', marker='o', markersize=4)
ax2.fill_between(df['data'], df['ipca_acumulado_12_meses'], alpha=0.3, color='#ff7f0e')
ax2.axhline(y=df['ipca_acumulado_12_meses'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df["ipca_acumulado_12_meses"].mean():.2f}%')
ax2.set_title('IPCA Acumulado 12 Meses', fontsize=14, fontweight='bold')
ax2.set_ylabel('IPCA 12M (%)')
ax2.set_xlabel('Data')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Periodos de maior inflacao

In [ ]:
# Top 5 maiores IPCA mensais
top5_maior = df.nlargest(5, 'ipca_mensal')[['data', 'ipca_mensal']]

print('TOP 5 - MAIORES IPCA MENSAIS:')
for idx, row in top5_maior.iterrows():
    print(f'{row["data"].strftime("%b/%Y")}: {row["ipca_mensal"]:.3f}%')

print()

# Top 5 menores IPCA mensais
top5_menor = df.nsmallest(5, 'ipca_mensal')[['data', 'ipca_mensal']]

print('TOP 5 - MENORES IPCA MENSAIS:')
for idx, row in top5_menor.iterrows():
    print(f'{row["data"].strftime("%b/%Y")}: {row["ipca_mensal"]:.3f}%')

## 5. Analise por ano

In [ ]:
# Adicionar coluna de ano
df['ano'] = df['data'].dt.year

# Agregar por ano
por_ano = df.groupby('ano').agg({
    'ipca_mensal': ['mean', 'min', 'max', 'sum'],
    'ipca_acumulado_12_meses': ['mean', 'min', 'max']
}).round(2)

print('IPCA POR ANO:')
print(por_ano)

## 6. Inflacao anual total

In [ ]:
# Calcular inflacao anual acumulada
inflacao_anual = df.groupby('ano')['ipca_mensal'].sum().round(2)

# Grafico de inflacao por ano
plt.figure(figsize=(12, 6))
plt.bar(inflacao_anual.index, inflacao_anual.values, color='#2ca02c', alpha=0.7, edgecolor='black')
plt.title('Inflacao Anual Acumulada (IPCA)', fontsize=14, fontweight='bold')
plt.xlabel('Ano')
plt.ylabel('IPCA Anual (%)')
plt.grid(True, alpha=0.3, axis='y')

# Adicionar valores nas barras
for i, v in enumerate(inflacao_anual.values):
    plt.text(inflacao_anual.index[i], v + 0.1, f'{v:.2f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Resumo da analise

In [ ]:
print('
RESUMO DA ANALISE DE INFLACAO:')
print('=' * 60)
print(f'Periodo: {df["data"].min().strftime("%b/%Y")} a {df["data"].max().strftime("%b/%Y")}')
print(f'Total de meses: {len(df)}')
print()
print(f'Inflacao media mensal: {df["ipca_mensal"].mean():.3f}%')
print(f'Inflacao media anual: {inflacao_anual.mean():.2f}%')
print()
print(f'Maior inflacao mensal: {df["ipca_mensal"].max():.3f}%')
print(f'Menor inflacao mensal: {df["ipca_mensal"].min():.3f}%')
print()
print('Dados prontos para proxima etapa')